# P2.11 · Text classifier on AG News (vocab, padding + mask, mean pooling, 1-layer transformer)

**ID:** P2.11 · **Target:** 35 min · **Plan week:** 4 · **Part:** P (PyTorch first principles)

**The drill.** Classify news headlines + descriptions into 4 topics (World, Sports, Business, Sci/Tech). Build the
text pipeline yourself: tokenizer → vocabulary with `<pad>`=0 and `<unk>`=1 → variable-length `Dataset` →
`collate_fn` that pads and returns a mask → two models sharing the same pipeline:
(A) `nn.Embedding(padding_idx=0)` + **masked mean pooling** (a fastText-style bag of embeddings), and
(B) the same plus learned positions and a **1-layer `nn.TransformerEncoder`** with `src_key_padding_mask`.

**Must include (from the plan):** tokenize, `nn.Embedding`, padding + mask, mean-pooling or 1-layer transformer
encoder (here: both). Also: a unit check that **masked mean pooling is invariant to extra padding** (and that
naive mean pooling is not), initial loss ≈ ln 4, overfit-one-batch, whole-set eval under `eval()` + `no_grad()`.

**Rules.** No docs, no LLM, no tokenizer libraries. Data: `DATA/"ag_news"/{train,test}.parquet` (columns
`text`, `label`). Subsample train for speed as specified. Each solution cell keeps only its spec.

## Sources
- X. Zhang, J. Zhao, Y. LeCun, *Character-level Convolutional Networks for Text Classification*, NeurIPS 2015,
  arXiv:1509.01626 — https://arxiv.org/abs/1509.01626 (introduces the AG News benchmark: 4 classes, 120k/7.6k).
- AG News dataset card (label ids 0 World, 1 Sports, 2 Business, 3 Sci/Tech): https://huggingface.co/datasets/fancyzhx/ag_news
- A. Joulin, E. Grave, P. Bojanowski, T. Mikolov, *Bag of Tricks for Efficient Text Classification* (fastText),
  EACL 2017, arXiv:1607.01759 — https://arxiv.org/abs/1607.01759 (averaged word embeddings + linear classifier).
- A. Vaswani et al., *Attention Is All You Need*, NeurIPS 2017, arXiv:1706.03762 — https://arxiv.org/abs/1706.03762
- PyTorch docs: `nn.Embedding` (`padding_idx`: that row gets no gradient)
  https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html ·
  `nn.TransformerEncoder` (`src_key_padding_mask`: True = ignore; `enable_nested_tensor`)
  https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html ·
  `nn.utils.rnn.pad_sequence` https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html
- PyTorch tutorial *Text classification with the torchtext library* (EmbeddingBag on AG News; the pipeline shape
  this drill rebuilds without torchtext): https://docs.pytorch.org/tutorials/beginner/text_sentiment_ngrams_tutorial.html

In [ ]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from prep import DATA, Session, check, check_close, seed_everything

s = Session("P2.11", target_min=35)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["World", "Sports", "Business", "Sci/Tech"]

In [ ]:
# 1. DATA. Read train/test parquet. Shuffle train with df.sample(frac=1, random_state=0); take the first 20 000 rows as
#    train_df and the next 5 000 as val_df; test_df = full test (7 600). Print class balance and 2 examples.
# YOUR CODE HERE


In [ ]:
# 2. TOKENIZER + VOCAB. tokenize(text) -> list[str]: lowercase, then re.findall(r"[a-z0-9]+(?:'[a-z]+)?", ...).
#    Build the vocab from TRAIN only: Counter over all train tokens; keep tokens with count >= 2, most frequent first,
#    at most 30 000 -> itos = ["<pad>", "<unk>", ...]; stoi = {tok: i}. PAD, UNK = 0, 1.
#    encode(text, max_len=128) -> list[int] (unknown -> UNK, truncate to max_len). Print vocab size, OOV rate on val.
# YOUR CODE HERE


In [ ]:
# 3. DATASET + COLLATE. `TextDataset(df)`: stores encoded LongTensors and labels; __getitem__ -> (ids_1d, label).
#    collate(batch) -> (ids (B, T) long padded with PAD to the batch's max length, mask (B, T) bool with
#    True = real token, labels (B,) long). Use nn.utils.rnn.pad_sequence(batch_first=True, padding_value=PAD).
#    DataLoaders: train batch 64 shuffle; val/test batch 512; all with collate_fn=collate.
# YOUR CODE HERE


In [ ]:
# 4. MODEL A: `MeanPoolClassifier(vocab_size, d=128, n_classes=4)`. forward(ids, mask) -> (B, 4) logits.
#    nn.Embedding(vocab_size, d, padding_idx=PAD) -> masked mean over time:
#    sum(E * mask[..., None], dim=1) / mask.sum(1, keepdim=True).clamp(min=1)  (an all-pad row must not give NaN)
#    -> Dropout(0.2) -> Linear(d, n_classes). Write the pooling as a standalone `masked_mean(x, mask)`.
# YOUR CODE HERE


In [ ]:
# 5. MODEL B: `TransformerClassifier(vocab_size, d=128, n_heads=4, max_len=128, n_classes=4)`.
#    token Embedding(padding_idx=PAD) + learned position Embedding(max_len, d) (positions 0..T-1);
#    ONE nn.TransformerEncoderLayer(d, n_heads, dim_feedforward=256, dropout=0.1, batch_first=True, norm_first=True)
#    wrapped in nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False);
#    call it with src_key_padding_mask = ~mask (True = PAD = ignore). Then LayerNorm, masked_mean, Linear.
# YOUR CODE HERE


In [ ]:
# 6. TRAIN / EVAL. evaluate(model, dl) -> (mean loss, accuracy) over the whole set (eval, no_grad).
#    fit(model, epochs, lr) -> history {"train_loss", "val_acc"}: AdamW (wd=1e-2), CE, clip grad-norm 1.0,
#    keep the best-val-acc state. Before training each model: record its initial val loss (≈ ln 4).
#    Overfit check: a fresh TransformerClassifier in eval mode, one train batch, Adam lr=1e-3, 200 steps ->
#    `overfit_loss` < 0.05.
#    model_mean = fit(MeanPoolClassifier, 5 epochs, lr=3e-3); model_tf = fit(TransformerClassifier, 5 epochs, lr=1e-3).
#    Store `init_losses = {"mean": ..., "tf": ...}`, `val_accs = {...}` (best), test accuracy `test_accs = {...}` (once).
# YOUR CODE HERE


## Self-check
Expects: `tokenize`, `itos`, `stoi`, `encode`, `PAD`, `UNK`, `collate`, `train_ds`, `masked_mean`, `model_mean`,
`model_tf` (both `forward(ids, mask) -> (B, 4)`), `init_losses`, `overfit_loss`, `val_accs`, `test_accs`.
The padding-invariance check appends 7 PAD columns to a real batch and requires identical logits.

In [ ]:
check("special tokens", itos[PAD] == "<pad>" and itos[UNK] == "<unk>" and PAD == 0 and UNK == 1)
check("unknown words map to <unk>", encode("qwxzvjk blorptastic") == [UNK, UNK])
_train_toks = {t for txt in train_df.text for t in tokenize(txt)}
_test_only = {t for txt in test_df.text for t in tokenize(txt)} - _train_toks
check("vocab built from train only (no test-only words)", len(_test_only) > 0 and not any(w in stoi for w in _test_only))
batch = [train_ds[i] for i in range(5)]
ids, mask, y = collate(batch)
check("collate shapes", ids.shape == mask.shape and ids.shape[0] == 5 and ids.shape[1] == max(len(b[0]) for b in batch))
check("mask marks real tokens", torch.equal(mask.sum(1), torch.tensor([len(b[0]) for b in batch])) and mask.dtype == torch.bool)
check("padding uses PAD", (ids[~mask] == PAD).all().item())
x = torch.randn(3, 6, 4)
mk = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0]], dtype=torch.bool)
check_close("masked_mean row 0", masked_mean(x, mk)[0], x[0, :3].mean(0))
check("masked_mean all-pad row is finite", torch.isfinite(masked_mean(x, mk)).all().item())


def pad_more(ids, mask, k=7):
    return (torch.cat([ids, torch.full((ids.shape[0], k), PAD, dtype=ids.dtype)], 1),
            torch.cat([mask, torch.zeros(mask.shape[0], k, dtype=torch.bool)], 1))


ids, mask, _ = next(iter(val_dl))
ids, mask = ids[:64], mask[:64]
ids2, mask2 = pad_more(ids, mask)
for name, m in [("mean-pool", model_mean), ("transformer", model_tf)]:
    m.eval()
    emb = next(mm for mm in m.modules() if isinstance(mm, nn.Embedding))
    check(f"{name}: token Embedding has padding_idx=0", emb.padding_idx == PAD)
    with torch.no_grad():
        a = m(ids.to(dev), mask.to(dev))
        b = m(ids2.to(dev), mask2.to(dev))
    check(f"{name}: logits (B, 4)", a.shape == (64, 4))
    check_close(f"{name}: invariant to extra padding", b, a, atol=1e-4, rtol=1e-4)
with torch.no_grad():
    e = next(mm for mm in model_mean.modules() if isinstance(mm, nn.Embedding))
    naive = lambda i: e(i.to(dev)).mean(1)  # mean over ALL positions, pads included
    check("naive (unmasked) mean is NOT padding-invariant", not torch.allclose(naive(ids), naive(ids2), atol=1e-4))
for k in ("mean", "tf"):
    check(f"{k}: initial loss ~ ln 4", abs(init_losses[k] - math.log(4)) < 0.2, f"got {init_losses[k]:.3f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
check("mean-pool test acc > 0.85", test_accs["mean"] > 0.85, f"got {test_accs['mean']:.3f}")
check("transformer test acc > 0.80", test_accs["tf"] > 0.80, f"got {test_accs['tf']:.3f}")

## Say it out loud (P3)
1. Why build the vocabulary on train only? What happens to test words that are not in it?
2. What does `padding_idx=0` do (forward and backward)? Is it enough on its own to make pooling padding-safe?
3. Why must the mean divide by the number of *real* tokens? What breaks if you divide by T?
4. `src_key_padding_mask`: which value means "ignore"? What would go wrong if you passed `mask` instead of `~mask`?
5. Why does the transformer need position embeddings while the mean-pool model doesn't? Does the transformer beat the bag of embeddings here, and why might it not?
6. Why pad per batch in `collate_fn` instead of padding everything to the global max length? What further trick reduces padding?
7. Where would `ignore_index` come in if this were a token-level task (e.g., tagging or LM)?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up